In [5]:
import json
import numpy as np
from sentence_transformers import SentenceTransformer
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_core.documents import Document
from transformers import AutoTokenizer, AutoModel

In [2]:
## Load chunks
with open(r"..\data\chunked\chunks_type1.json") as f:
    chunks = json.load(f)

In [3]:
## creating document object
documents = []

for c in chunks:
    doc = Document(
        page_content=c["text"],
        metadata={"company_name": c["company_name"], "section": c["section"], "chunk_id": c["chunk_id"]},
    )
    documents.append(doc)

In [8]:
## embedding model intialize
embedding_model = HuggingFaceEmbeddings(
    model_name="intfloat/e5-large"
)

# embedding_model = AutoModel.from_pretrained("intfloat/e5-large")


c:\Users\ansul\OneDrive\Desktop\data science project\financial_rag\.venv\Lib\site-packages\huggingface_hub\file_download.py:129: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\ansul\.cache\huggingface\hub\models--intfloat--e5-large. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
Loading weights: 100%|██████████| 391/391 [00:00<00:00, 4004.45it/s]
BertModel

## Creating index file for first time

In [9]:
## building vector store
vectorstore = FAISS.from_documents(
    documents,
    embedding_model
)

vectorstore.save_local(r"..\data\embeddings_E5_Large")

## Use the local vector index

In [10]:
## loading the vector store
vectorstore = FAISS.load_local(
    r"..\data\embeddings",embeddings=embedding_model,allow_dangerous_deserialization=True
)

## Testing Retrival 

In [10]:
vectorstore.similarity_search("how does companies stock price depend on global and regional economic conditions ?", k=3)

[Document(id='8f26103a-5bdd-4031-984f-5a2bd45dd4b6', metadata={'company_name': 'ORACLE', 'section': 'risk_factors', 'chunk_id': 25}, page_content=', our stock repurchase program may not enhance long-term stockholder value. In addition, the Inflation Reduction Act, signed into law on August 16, 2022, imposes an excise tax of 1% (potentially increasing to 4% under certain U.S. tax proposals) on certain corporate stock repurchases. 30 Table of Contents Index to Financial Statements General Risks Economic, political and market conditions can adversely affect our business, results of operations and financial condition, including our revenue growth and profitability, which in turn could adversely affect our stock price. Our business is influenced by a range of factors that are beyond our control and that we have no comparative advantage in forecasting. These include: general economic and business conditions; overall demand for enterprise cloud, license and hardware products and services; gov

In [11]:
vectorstore.similarity_search("What is the risks associated with netflix?", k=3)

[Document(id='7d44c81d-4866-4ea3-ab62-ba5cecb0798f', metadata={'company_name': 'NFLX', 'section': 'risk_factors', 'chunk_id': 5}, page_content=" which compete directly with us or have investments in competing streaming content providers. In many instances, our agreements also include provisions by which the partner bills consumers directly for the Netflix service or otherwise offers services or products in connection with offering our service. If partners or other providers do a better job of connecting consumers with content they want to watch, for example through multi-service discovery interfaces, our service may be adversely impacted. We intend to continue to broaden our relationships with existing partners and to increase our capability to stream TV series and films to other platforms and partners over time. If we are not successful in maintaining existing and creating new relationships, or if we encounter technological, content licensing, regulatory, business or other impediments